In [ ]:
%pip install -q kagglehub tables pyyaml torchinfo

import os

# Номер физической GPU на сервере.
# Перед сменой номера перезапусти kernel.
PHYSICAL_GPU = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = PHYSICAL_GPU

import kagglehub

path = kagglehub.dataset_download("liuxu77/largest")

print("Датасет скачан в:", path)
print("Выбрана физическая GPU:", PHYSICAL_GPU)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Ноутбук должен лежать в корне репозитория STAEformer.
repo_dir = Path.cwd().resolve()

if not (repo_dir / "model").exists():
    raise RuntimeError(
        "Запусти ноутбук из корня репозитория STAEformer: "
        "рядом должны находиться папки model, lib и data."
    )

print("Корень проекта:", repo_dir)


In [ ]:
root = Path(path)

files = {
    file.name: file
    for file in root.rglob("*")
    if file.is_file()
}

print("Найдено файлов:", len(files))


In [ ]:
YEAR = 2019

output_dir = repo_dir / "data" / "SD"
output_dir.mkdir(parents=True, exist_ok=True)

ca_meta = pd.read_csv(files["ca_meta.csv"])

sd_meta = (
    ca_meta[ca_meta["District"] == 11]
    .copy()
    .reset_index(drop=True)
)

assert len(sd_meta) == 716, (
    f"Ожидалось 716 датчиков SD, получено {len(sd_meta)}"
)

sd_meta.to_csv(output_dir / "sd_meta.csv", index=False)

print("Количество датчиков SD:", len(sd_meta))
print("Папка данных:", output_dir)


In [ ]:
history_path = files[f"ca_his_raw_{YEAR}.h5"]

sd_sensor_ids = sd_meta["ID"].astype(str).tolist()

ca_his = pd.read_hdf(history_path)
ca_his.columns = ca_his.columns.astype(str)

sd_his = ca_his.loc[:, sd_sensor_ids].copy()

sd_his.index = pd.to_datetime(sd_his.index)
sd_his = sd_his.sort_index()
sd_his = sd_his.astype(np.float32)

missing_before = int(sd_his.isna().sum().sum())

sd_his = sd_his.fillna(0)

print("Форма SD:", sd_his.shape)

In [ ]:
values = sd_his.to_numpy(dtype=np.float32)

num_steps, num_nodes = values.shape

traffic = values[..., np.newaxis]

time_of_day = (
    (sd_his.index.hour * 60 + sd_his.index.minute)
    / (24 * 60)
).to_numpy(dtype=np.float32)

time_of_day = np.tile(
    time_of_day[:, np.newaxis, np.newaxis],
    (1, num_nodes, 1)
)

day_of_week = sd_his.index.dayofweek.to_numpy(dtype=np.float32)

day_of_week = np.tile(
    day_of_week[:, np.newaxis, np.newaxis],
    (1, num_nodes, 1)
)

data = np.concatenate(
    [traffic, time_of_day, day_of_week],
    axis=-1
)

print("sd_his:", sd_his.shape)
print("data:", data.shape)


In [ ]:
num_steps = data.shape[0]

indices = []

for start in range(num_steps - 12 - 12 + 1):
    x_end = start + 12
    y_end = x_end + 12
    indices.append([start, x_end, y_end])

indices = np.array(indices, dtype=np.int64)

print("Форма indices:", indices.shape)
print("Первое окно:", indices[0])
print("Последнее окно:", indices[-1])

In [ ]:
num_samples = len(indices)

num_train = round(num_samples * 0.6)
num_val = round(num_samples * 0.2)

train_indices = indices[:num_train]
val_indices = indices[num_train:num_train + num_val]
test_indices = indices[num_train + num_val:]

In [ ]:
np.savez_compressed(
    output_dir / "data.npz",
    data=data.astype(np.float32)
)

np.savez_compressed(
    output_dir / "index.npz",
    train=train_indices,
    val=val_indices,
    test=test_indices
)

print("Данные сохранены:")
for file in sorted(output_dir.iterdir()):
    print(" -", file.name)


In [ ]:
required_project_paths = [
    repo_dir / "model" / "train.py",
    repo_dir / "model" / "STAEformer.yaml",
    repo_dir / "lib",
]

missing = [str(path) for path in required_project_paths if not path.exists()]

if missing:
    raise FileNotFoundError(
        "В репозитории не хватает файлов:\n" + "\n".join(missing)
    )

print("Файлы репозитория найдены.")


In [ ]:
required_data_files = [
    output_dir / "data.npz",
    output_dir / "index.npz",
]

for file in required_data_files:
    if not file.exists():
        raise FileNotFoundError(file)

print("Датасет готов к запуску:")
for file in required_data_files:
    print(f" - {file.name}: {file.stat().st_size / 1024**2:.2f} MB")


In [ ]:
import yaml

config_path = repo_dir / "model" / "STAEformer.yaml"

with config_path.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

print("Разделы конфига:", config.keys())


In [ ]:
if "SD" not in config:
    raise KeyError(
        "В model/STAEformer.yaml нет раздела SD."
    )

display(config["SD"])


In [ ]:
!nvidia-smi


In [ ]:
import torch

print("CUDA доступна:", torch.cuda.is_available())
print("Видимых GPU:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU внутри ноутбука:", torch.cuda.get_device_name(0))


In [ ]:
print("Команда запуска:")
print("cd model && python -u train.py -d SD -g 0")


In [ ]:
!cd model && python -u train.py -d SD -g 0
